In [1]:
#> imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import astropy.io.fits as fits
from astropy.table import Table

import gaia.astrometry as astrometry
import gaia.geometry as geometry
import gaia.order as order

In [2]:
#> file
file = './gaia/gaia_full_table.fit'

#> position table
table = Table.read(file, hdu=1)
df = table.to_pandas()

#> reducing to quads
df = df[ df['Type'] == b'Quad  ' ]

#> getting keys
print(f'> There are {len(df.keys())} keys: {list(df.keys())}')

#> binary to normal
for key in df.keys():
    try:
        df[key] = [ x.decode('utf-8') for x in df[key] ]
    except:
        continue

df = df.sort_values(['Name', 'Comp']).reset_index(drop=True)

#> printing
df.describe()
df

> There are 27 keys: ['TimeDelay', 'Name', 'Comp', 'RAJ2000', 'DEJ2000', 'Type', 'MaxSep', 'rpos', 'BibCode', 'RApdeg', 'DEpdeg', 'RAcdeg', 'DEcdeg', 'zsource', 'zdef', 'zmilli', 'zqsocdr3', 'zSHSRC', 'GLens', 'CompId', 'RACodeg', 'e_RACodeg', 'DECodeg', 'e_DECodeg', 'GaiaDR3', 'AllWISE', 'recno']


,TimeDelay,Name,Comp,RAJ2000,DEJ2000,Type,MaxSep,rpos,BibCode,RApdeg,...,zSHSRC,GLens,CompId,RACodeg,e_RACodeg,DECodeg,e_DECodeg,GaiaDR3,AllWISE,recno
0,b'*',2MASSJ11344050-2103230,A,173.669102,-21.056434,Quad,3.682,Gaia_DR3,2018MNRAS.476..927L,173.669101,...,NaN,NaN,0,NaN,NaN,NaN,NaN,3541826024524572544,NaN,568
1,b'*',2MASSJ11344050-2103230,B,173.669319,-21.055947,Quad,3.682,Gaia_DR3,2018MNRAS.476..927L,173.669319,...,NaN,NaN,0,NaN,NaN,NaN,NaN,3541826024526317568,NaN,569
2,b'*',2MASSJ11344050-2103230,C,173.668522,-21.056649,Quad,3.682,Gaia_DR3,2018MNRAS.476..927L,173.668521,...,NaN,NaN,0,NaN,NaN,NaN,NaN,3541826024526317312,NaN,565
3,b'*',2MASSJ11344050-2103230,D,173.668731,-21.056055,Quad,3.682,Gaia_DR3,2018MNRAS.476..927L,173.668728,...,NaN,NaN,0,NaN,NaN,NaN,NaN,3541826024524572288,NaN,566
4,b'*',2MASSJ11344050-2103230,G,173.668878,-21.056218,Quad,3.682,HST,2018MNRAS.476..927L,173.668878,...,NaN,NaN,0,NaN,NaN,NaN,NaN,-9223372036854775808,NaN,567
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
369,b'*',WISE025942.9-163543,A,44.928792,-16.595361,Quad,1.575,Gaia_DR3,2018RNAAS...2...21S,44.928792,...,NaN,NaN,0,NaN,NaN,NaN,NaN,5153828508862119040,NaN,153
370,b'*',WISE025942.9-163543,B,44.928700,-16.595117,Quad,1.575,Gaia_DR3,2018RNAAS...2...21S,44.928699,...,NaN,NaN,0,NaN,NaN,NaN,NaN,5153828504567283072,NaN,152
371,b'*',WISE025942.9-163543,C,44.928365,-16.595205,Quad,1.575,Gaia_DR3,2018RNAAS...2...21S,44.928363,...,NaN,NaN,0,NaN,NaN,NaN,NaN,5153828508862118912,NaN,149
372,b'*',WISE025942.9-163543,D,44.928584,-16.595469,Quad,1.575,HST,2018RNAAS...2...21S,44.928584,...,NaN,NaN,0,NaN,NaN,NaN,NaN,-9223372036854775808,NaN,150


In [3]:
#> iterating through each lens name
names = np.unique(df['Name'].to_numpy())
comps = np.unique(df['Comp'])
components = np.zeros(shape=(len(names), len(comps)), dtype=int)
redshifts = np.zeros(shape=(len(names), 2))
rpos = np.zeros(shape=(len(names), 1))
for i, name in enumerate(names):

    
    gal = df[ df['Name'] == name ] # selecting galaxy
    comp = gal['Comp'].to_numpy()  # getting components
    r = np.unique(gal['rpos'])

    #> getting redshifts
    zl = np.unique(gal['zdef'])
    zs = np.unique(gal['zsource'])
    zl = zl[~np.isnan(zl)]
    zs = zs[~np.isnan(zs)]
    if len(zl) == 0: zl = None
    else: zl=zl[0]
    if len(zs) == 0: zs = None
    else: zs=zs[0]
    
    redshifts[i][0] = zl
    redshifts[i][1] = zs

    for c in comp:
        indx = np.where(comps==c)
        components[i][indx] += 1

#> creating new dataframe
column_names = np.hstack([['Name'], comps, ['zl', 'zs']])
data = np.concatenate([names.reshape(len(names), 1), components, redshifts], axis=1)
gals = pd.DataFrame(data, columns=column_names)

#> adding flags
gals['n_defs'] = gals['G '].apply(int) + gals['G1'].apply(int) + gals['G2'].apply(int) + gals['G3'].apply(int) + gals['G4'].apply(int) + gals['G5'].apply(int)
gals['n_ims'] = gals['A '].apply(int) + gals['A1'].apply(int) + gals['A2'].apply(int) + gals['B '].apply(int) + gals['C '].apply(int) + gals['D '].apply(int) + gals['E '].apply(int) + gals['F '].apply(int)
print(gals.to_string())

                        Name A  A1 A2 B  C  D  E  F  G  G1 G2 G3 G4 G5 S       zl      zs  n_defs  n_ims
0     2MASSJ11344050-2103230  1  0  0  1  1  1  0  0  1  0  0  0  0  0  0     NaN    2.77       1      4
1     2MASSJ13102005-1714579  1  0  0  1  1  1  0  0  0  1  1  0  0  0  0   0.293   1.975       2      4
2     2MASXJ01471020+4630433  1  0  0  1  1  1  0  0  1  0  0  0  0  0  0   0.678   2.377       1      4
3            ATLASJ2344-3056  1  0  0  1  1  1  0  0  1  0  0  0  0  0  0  0.9472  1.2978       1      4
4                  B0128+437  1  0  0  1  1  1  0  0  0  0  0  0  0  0  0     NaN   3.124       0      4
5                  B0712+472  1  0  0  1  1  1  0  0  1  0  0  0  0  0  0    0.41    1.34       1      4
6                  B1359+154  1  0  0  1  1  1  1  1  0  1  1  1  0  0  0    0.88   3.235       3      6
7                  B1422+231  1  0  0  1  1  1  0  0  1  0  0  0  0  0  0  0.3366    3.62       1      4
8                  B1555+375  1  0  0  1  1  1  0  0  1

In [4]:
#> getting one deflector quads
oneg_quads = gals[ (gals['n_defs'] == 1) & (gals['n_ims'] == 4) ]
oneg_quads = oneg_quads.reset_index(drop=True)

In [5]:
#> parsing the larger table based on their configuration
names = oneg_quads['Name'].to_numpy()
oneg_quads_comp = df[ df['Name'].map(lambda u: u in names) ].reset_index(drop=True)
oneg_quads_comp

all_data_sets = [oneg_quads]
comps_sets = []

for j, data_set in enumerate(all_data_sets):

    names = data_set['Name'].to_numpy()
    data_comp = df[ df['Name'].map(lambda u: u in names) ].reset_index(drop=True)

    # RA, Dec, zpra, zpdec
    all_arc_pos = None
    for i, name in enumerate(names):
    
        #> getting components
        lens_comps  = data_comp[ (data_comp['Name'] == name) ]  # images
        lens_galaxy = data_comp[ (data_comp['Name'] == name) & (data_comp['Comp'].map(lambda u: 'G' in u)) ] # galaxy
    
        #> component ra and decs
        if name in ['SDSS1138+0314', 'HE1113-0641']:
            comp_ra = lens_comps['RApdeg'].to_numpy()
            comp_dec = lens_comps['DEpdeg'].to_numpy()

            #> original does not match simbad--this is from simbad
            if name == 'SDSS1138+0314':
                comp_ra[0] = 174.515542	
                comp_dec[0] = 3.249389
            
            #> getting ra and decs
            gal_ra = lens_galaxy['RApdeg'].to_numpy()[0]
            gal_dec = lens_galaxy['DEpdeg'].to_numpy()[0]
            
        else:

            #> previously used: RAJ2000
            comp_ra = lens_comps['RAJ2000'].to_numpy()
            comp_dec = lens_comps['DEJ2000'].to_numpy()
            
            #> getting ra and decs
            gal_ra = lens_galaxy['RAJ2000'].to_numpy()[0]
            gal_dec = lens_galaxy['DEJ2000'].to_numpy()[0]
    
    
        #> converting
        x_arc, y_arc = astrometry.RADEC_to_ARCSEC(comp_ra, comp_dec, gal_ra, gal_dec)
        # x_arc = np.concatenate([x_arc, [0.0]])
        # y_arc = np.concatenate([y_arc, [0.0]])
        
        #> saving
        arc_pos = np.vstack([x_arc, y_arc]).T
        if i == 0: all_arc_pos = arc_pos
        else: all_arc_pos = np.vstack([all_arc_pos, arc_pos])
            
    if all_arc_pos is not None:
        data_comp['dRA'] = all_arc_pos[:,0]
        data_comp['dDEC'] = all_arc_pos[:,1]
        # data_comp.to_csv(f'gaia_{j+1}D_quads_positions_26D.csv', index=False)
        comps_sets.append(data_comp)

oneg_comp = comps_sets[0]
oneg_comp

,TimeDelay,Name,Comp,RAJ2000,DEJ2000,Type,MaxSep,rpos,BibCode,RApdeg,...,CompId,RACodeg,e_RACodeg,DECodeg,e_DECodeg,GaiaDR3,AllWISE,recno,dRA,dDEC
0,b'*',2MASSJ11344050-2103230,A,173.669102,-21.056434,Quad,3.682,Gaia_DR3,2018MNRAS.476..927L,173.669101,...,0,NaN,NaN,NaN,NaN,3541826024524572544,NaN,568,0.752181,-0.779
1,b'*',2MASSJ11344050-2103230,B,173.669319,-21.055947,Quad,3.682,Gaia_DR3,2018MNRAS.476..927L,173.669319,...,0,NaN,NaN,NaN,NaN,3541826024526317568,NaN,569,1.481036,0.976
2,b'*',2MASSJ11344050-2103230,C,173.668522,-21.056649,Quad,3.682,Gaia_DR3,2018MNRAS.476..927L,173.668521,...,0,NaN,NaN,NaN,NaN,3541826024526317312,NaN,565,-1.195462,-1.552
3,b'*',2MASSJ11344050-2103230,D,173.668731,-21.056055,Quad,3.682,Gaia_DR3,2018MNRAS.476..927L,173.668728,...,0,NaN,NaN,NaN,NaN,3541826024524572288,NaN,566,-0.495545,0.587
4,b'*',2MASSJ11344050-2103230,G,173.668878,-21.056218,Quad,3.682,HST,2018MNRAS.476..927L,173.668878,...,0,NaN,NaN,NaN,NaN,-9223372036854775808,NaN,567,0.000000,0.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
240,b'*',WISE025942.9-163543,A,44.928792,-16.595361,Quad,1.575,Gaia_DR3,2018RNAAS...2...21S,44.928792,...,0,NaN,NaN,NaN,NaN,5153828508862119040,NaN,153,0.693842,-0.301
241,b'*',WISE025942.9-163543,B,44.928700,-16.595117,Quad,1.575,Gaia_DR3,2018RNAAS...2...21S,44.928699,...,0,NaN,NaN,NaN,NaN,5153828504567283072,NaN,152,0.376630,0.576
242,b'*',WISE025942.9-163543,C,44.928365,-16.595205,Quad,1.575,Gaia_DR3,2018RNAAS...2...21S,44.928363,...,0,NaN,NaN,NaN,NaN,5153828508862118912,NaN,149,-0.778177,0.261
243,b'*',WISE025942.9-163543,D,44.928584,-16.595469,Quad,1.575,HST,2018RNAAS...2...21S,44.928584,...,0,NaN,NaN,NaN,NaN,-9223372036854775808,NaN,150,-0.023000,-0.691


In [20]:
#> getting quad with one deflector and their image positions
oneg_comp = comps_sets[0]

#> lensing observables
observables = ['t12', 't23', 't34', 'd2/d1', 'd3/d1', 'd4/d1', 'dt23', 'd01', 'd02', 'd03', 'd04']
columns = ['Name'] + observables + ['arrival_order']

#> dataframe
obs = pd.DataFrame(columns=columns)

#> iterating through each lens
for i, name in enumerate(np.unique(oneg_comp['Name'])):

    #> getting images
    xims = oneg_comp[ oneg_comp['Name'] == name ]['dRA'].to_numpy()[:-1]
    yims = oneg_comp[ oneg_comp['Name'] == name ]['dDEC'].to_numpy()[:-1]
    images = np.vstack([xims, yims]).T

    #> arrival sequence
    ordered = order.arrivalOrder(images)

    #> saving sequence
    inA = np.where(ordered==images[0])[0][0]
    inB = np.where(ordered==images[1])[0][0]
    inC = np.where(ordered==images[2])[0][0]
    inD = np.where(ordered==images[3])[0][0]
    arrival_order = np.full(shape=(4), fill_value='')
    arrival_order[inA] = 'A'
    arrival_order[inB] = 'B'
    arrival_order[inC] = 'C'
    arrival_order[inD] = 'D'
    arrival_order = ''.join(arrival_order)

    # plot(name, ordered)
    # print(name, arrival_order)

    # arrival_order = 'XXXX'

    #> lensing observables
    lensObs = geometry.lensObs((0,0), ordered, observables)

    row = [name]
    for x in lensObs:
        if np.ndim(x) == 0: row.append(x)
        else: row.append(np.asarray(x).flatten()[0])
    row.append(arrival_order)

    #> saving
    obs.loc[i] = row
# obs.to_csv('gaia_obs_quads.csv', index=False)
obs

,Name,t12,t23,t34,d2/d1,d3/d1,d4/d1,dt23,d01,d02,d03,d04,arrival_order
0,2MASSJ11344050-2103230,160.990859,79.388255,176.174471,0.905398,0.552758,0.392132,1.421837,1.959039,1.773709,1.082875,0.768202,CBAD
1,2MASXJ01471020+4630433,67.496249,33.282576,176.414406,1.053678,0.974821,0.551357,4.304327,2.131754,2.246182,2.078077,1.175357,CBAD
2,ATLASJ2344-3056,162.337535,78.863321,177.402246,1.037732,0.986751,0.773852,-0.623941,0.528391,0.548328,0.521390,0.408896,BDCA
3,B0712+472,79.388131,11.777773,161.516727,0.998735,0.994495,0.575153,-4.345365,0.821000,0.819962,0.816480,0.472200,DABC
4,B1422+231,75.739283,29.040993,180.458461,0.930297,0.895670,0.245843,-9.364461,1.086350,1.010628,0.973011,0.267072,CABD
5,B1555+375,84.086977,16.737991,158.935955,0.888588,0.926361,0.330377,1.005672,0.327466,0.290982,0.303351,0.108187,CABD
6,B1608+656,115.410051,37.399853,163.846015,0.720885,0.802417,0.760000,-3.608825,1.318184,0.950258,1.057732,1.001820,BACD
7,COSMOS5921+0638,105.996296,42.394842,164.428572,0.977629,0.994530,0.944665,6.551909,0.730169,0.713834,0.726175,0.689765,CABD
8,DESJ0029-3814,179.533147,51.227799,107.062330,0.650409,0.423661,0.426003,-1.753763,1.339179,0.871015,0.567359,0.570494,CABD
9,DESJ0053-2012,110.275839,33.431070,146.156380,1.028528,0.874074,0.334988,14.647397,1.839933,1.892422,1.608237,0.616356,ACBD


#### ***Opening & Editing Main File***
---

The all_quads_positions.csv file contains a combination of quad positions from a few different sources, mostly from GAIA and Liliya. Combining them is not a simple task and was mostly done by hand. However, the GAIA data needs to be able to be updated. To do this, I am going to open the original csv and update only the GAIA data, leaving the rest as entered (i.e., Liliya and SLED). 

I need to match the names from GAIA, then match the columns for this new file.

In [21]:
#> comparing files
position_file = 'all_quads_positions.csv'
main_df = pd.read_csv(position_file)
keys = list(main_df.keys())
print(keys)
main_df

['Unnamed: 0.1', 'Name', 'Comp', 'Unnamed: 0', 'alias', 'From', 'dRA', 'dDEC', 'edDEC', 'edRA', 'RAJ2000', 'DEJ2000', 'eRAJ2000', 'eDEJ2000', 'rpos', 'BibCode', 'RApdeg', 'DEpdeg', 'zsource', 'zdef', 'ezsource', 'ezdef', 'zsBibCode', 'zlBibCode', 'arrival_order']


,Unnamed: 0.1,Name,Comp,Unnamed: 0,alias,From,dRA,dDEC,edDEC,edRA,...,BibCode,RApdeg,DEpdeg,zsource,zdef,ezsource,ezdef,zsBibCode,zlBibCode,arrival_order
0,0,2MASSJ11344050-2103230,A,0,-,2026 Ducourant,0.7521809578509628,-0.7790000000071018,-,-,...,2018MNRAS.476..927L,173.66910082,-21.05643527,2.77,-,-,-,-,-,XXXX
1,1,2MASSJ11344050-2103230,B,1,-,2026 Ducourant,1.4810360922449488,0.9759999999999991,-,-,...,2018MNRAS.476..927L,173.66931899,-21.05594666,2.77,-,-,-,-,-,XXXX
2,2,2MASSJ11344050-2103230,C,2,-,2026 Ducourant,-1.1954620556420334,-1.5520000000037726,-,-,...,2018MNRAS.476..927L,173.66852069,-21.05665055,2.77,-,-,-,-,-,XXXX
3,3,2MASSJ11344050-2103230,D,3,-,2026 Ducourant,-0.4955447977212434,0.5869999999930542,-,-,...,2018MNRAS.476..927L,173.66872786,-21.0560536,2.77,-,-,-,-,-,XXXX
4,4,2MASSJ11344050-2103230,G,4,-,2026 Ducourant,0.0,0.0,-,-,...,2018MNRAS.476..927L,173.66887847,-21.05621777,2.77,-,-,-,-,-,XXXX
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
330,330,J1330-0905,A,330,Persephone's Torch,SLED,0,0,-,-,...,https://arxiv.org/abs/2604.13152,-,-,2.2245,-,0.0005,-,-,-,-
331,331,J1330-0905,B,331,Persephone's Torch,SLED,-0.4911,-0.0437,-,-,...,https://arxiv.org/abs/2604.13152,-,-,2.2245,-,0.0005,-,-,-,-
332,332,J1330-0905,C,332,Persephone's Torch,SLED,-0.2423,-0.0918,-,-,...,https://arxiv.org/abs/2604.13152,-,-,2.2245,-,0.0005,-,-,-,-
333,333,J1330-0905,D,333,Persephone's Torch,SLED,-0.2845,0.7672,-,-,...,https://arxiv.org/abs/2604.13152,-,-,2.2245,-,0.0005,-,-,-,-


In [22]:
#> getting only gaia data that is kept in the new file, and the relative and absolute positions
main_df_names = main_df['Name'].unique()
previous_columns = ['Name', 'Comp', 'dRA', 'dDEC', 'RAJ2000', 'DEJ2000', 'RApdeg', 'DEpdeg']
gaia_df = oneg_comp[ oneg_comp['Name'].apply(lambda x: x in main_df_names)].copy()[previous_columns]

#> getting arrival order for these
obs_order = obs[['Name', 'arrival_order']]
gaia_df = pd.merge(gaia_df, obs_order, how='left', on='Name')
gaia_df

,Name,Comp,dRA,dDEC,RAJ2000,DEJ2000,RApdeg,DEpdeg,arrival_order
0,2MASSJ11344050-2103230,A,0.752181,-0.779,173.669102,-21.056434,173.669101,-21.056435,CBAD
1,2MASSJ11344050-2103230,B,1.481036,0.976,173.669319,-21.055947,173.669319,-21.055947,CBAD
2,2MASSJ11344050-2103230,C,-1.195462,-1.552,173.668522,-21.056649,173.668521,-21.056651,CBAD
3,2MASSJ11344050-2103230,D,-0.495545,0.587,173.668731,-21.056055,173.668728,-21.056054,CBAD
4,2MASSJ11344050-2103230,G,0.000000,0.000,173.668878,-21.056218,173.668878,-21.056218,CBAD
...,...,...,...,...,...,...,...,...,...
225,WISE025942.9-163543,A,0.693842,-0.301,44.928792,-16.595361,44.928792,-16.595362,CADB
226,WISE025942.9-163543,B,0.376630,0.576,44.928700,-16.595117,44.928699,-16.595118,CADB
227,WISE025942.9-163543,C,-0.778177,0.261,44.928365,-16.595205,44.928363,-16.595207,CADB
228,WISE025942.9-163543,D,-0.023000,-0.691,44.928584,-16.595469,44.928584,-16.595470,CADB


In [38]:
#> updated columns
updated_columns = ['dRA', 'dDEC', 'RAJ2000', 'DEJ2000', 'RApdeg', 'DEpdeg', 'arrival_order']

#> cross-over keys
keys = ['Name', 'Comp']

#> setting index to keys 
main_indexed = main_df.set_index(keys)
gaia_indexed = gaia_df.set_index(keys)

#> finding intersection of dataframes (matching indexes)
matching_keys = main_indexed.index.intersection(gaia_indexed.index)

#> updating columns for matched indexes
main_indexed.loc[matching_keys, updated_columns] = (gaia_indexed.loc[matching_keys, updated_columns])

#> resetting index
main_df = main_indexed.reset_index()
#> saving
main_df.to_csv(position_file, index=False)

main_df

,Name,Comp,Unnamed: 0.1,Unnamed: 0,alias,From,dRA,dDEC,edDEC,edRA,...,BibCode,RApdeg,DEpdeg,zsource,zdef,ezsource,ezdef,zsBibCode,zlBibCode,arrival_order
0,2MASSJ11344050-2103230,A,0,0,-,2026 Ducourant,0.752181,-0.779,-,-,...,2018MNRAS.476..927L,173.669101,-21.056435,2.77,-,-,-,-,-,CBAD
1,2MASSJ11344050-2103230,B,1,1,-,2026 Ducourant,1.481036,0.976,-,-,...,2018MNRAS.476..927L,173.669319,-21.055947,2.77,-,-,-,-,-,CBAD
2,2MASSJ11344050-2103230,C,2,2,-,2026 Ducourant,-1.195462,-1.552,-,-,...,2018MNRAS.476..927L,173.668521,-21.056651,2.77,-,-,-,-,-,CBAD
3,2MASSJ11344050-2103230,D,3,3,-,2026 Ducourant,-0.495545,0.587,-,-,...,2018MNRAS.476..927L,173.668728,-21.056054,2.77,-,-,-,-,-,CBAD
4,2MASSJ11344050-2103230,G,4,4,-,2026 Ducourant,0.0,0.0,-,-,...,2018MNRAS.476..927L,173.668878,-21.056218,2.77,-,-,-,-,-,CBAD
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
330,J1330-0905,A,330,330,Persephone's Torch,SLED,0,0,-,-,...,https://arxiv.org/abs/2604.13152,-,-,2.2245,-,0.0005,-,-,-,-
331,J1330-0905,B,331,331,Persephone's Torch,SLED,-0.4911,-0.0437,-,-,...,https://arxiv.org/abs/2604.13152,-,-,2.2245,-,0.0005,-,-,-,-
332,J1330-0905,C,332,332,Persephone's Torch,SLED,-0.2423,-0.0918,-,-,...,https://arxiv.org/abs/2604.13152,-,-,2.2245,-,0.0005,-,-,-,-
333,J1330-0905,D,333,333,Persephone's Torch,SLED,-0.2845,0.7672,-,-,...,https://arxiv.org/abs/2604.13152,-,-,2.2245,-,0.0005,-,-,-,-


#### ***Updating Obs File***
---
The positions file has been updated, now it is time to update the all_obs_quad file!

In [39]:
#> reading previous file
obs_file = 'all_obs_quads.csv'
obs_df = pd.read_csv(obs_file)
obs_df

,Name,t12,t23,t34,d2/d1,d3/d1,d4/d1,dt23,d01,d02,d03,d04,d3/d2
0,1810.01633,148.668900,61.229272,157.392502,0.929732,0.747339,0.545220,4.266853,2.091220,1.944274,1.562850,1.140175,0.803822
1,2MASSJ11344050-2103230,160.990859,79.388255,176.174471,0.905398,0.552758,0.392132,1.421837,1.959039,1.773709,1.082875,0.768202,0.610514
2,2MASXJ01471020+4630433,67.496249,33.282576,176.414406,1.053678,0.974821,0.551357,4.304327,2.131754,2.246182,2.078077,1.175357,0.925160
3,ATLASJ2344-3056,162.337535,78.863321,177.402246,1.037732,0.986751,0.773852,-0.623941,0.528391,0.548328,0.521390,0.408896,0.950872
4,B0128+437,166.642462,33.726889,114.638397,0.773541,0.743710,0.609925,-9.702828,0.311082,0.240635,0.231355,0.189737,0.961436
...,...,...,...,...,...,...,...,...,...,...,...,...,...
61,WFI2026-4536,167.866845,30.079459,111.735674,0.779213,0.666326,0.756942,-12.914943,0.824742,0.642650,0.549547,0.624282,0.855127
62,WFI2033-4723,134.763345,33.443944,122.518557,0.904061,0.809099,0.587003,16.479074,1.440477,1.302279,1.165489,0.845564,0.894961
63,WG0214-2105,172.587701,68.865969,145.655630,0.890399,0.788669,0.802560,1.926882,0.981781,0.874176,0.774300,0.787937,0.885748
64,WG2100-4452,124.692279,23.198296,137.406381,0.943367,0.912157,0.770706,1.836102,1.434103,1.352885,1.308127,1.105272,0.966917


In [40]:
#> cross-over keys
keys = ['Name']

#> setting index to keys 
old_indexed = obs_df.set_index(keys)
new_indexed = obs.set_index(keys)

#> finding intersection of dataframes (matching indexes)
matching_keys = old_indexed.index.intersection(new_indexed.index)

#> updating columns for matched indexes
old_indexed.loc[matching_keys, observables] = (new_indexed.loc[matching_keys, observables])

#> resetting index
obs_df = old_indexed.reset_index()
obs_df['d3/d2'] = obs_df['d3/d1'] / obs_df['d2/d1']

#> saving
obs_df.to_csv(obs_file, index=False)

obs_df

,Name,t12,t23,t34,d2/d1,d3/d1,d4/d1,dt23,d01,d02,d03,d04,d3/d2
0,1810.01633,148.668900,61.229272,157.392502,0.929732,0.747339,0.545220,4.266853,2.091220,1.944274,1.562850,1.140175,0.803822
1,2MASSJ11344050-2103230,160.990859,79.388255,176.174471,0.905398,0.552758,0.392132,1.421837,1.959039,1.773709,1.082875,0.768202,0.610514
2,2MASXJ01471020+4630433,67.496249,33.282576,176.414406,1.053678,0.974821,0.551357,4.304327,2.131754,2.246182,2.078077,1.175357,0.925160
3,ATLASJ2344-3056,162.337535,78.863321,177.402246,1.037732,0.986751,0.773852,-0.623941,0.528391,0.548328,0.521390,0.408896,0.950872
4,B0128+437,166.642462,33.726889,114.638397,0.773541,0.743710,0.609925,-9.702828,0.311082,0.240635,0.231355,0.189737,0.961436
...,...,...,...,...,...,...,...,...,...,...,...,...,...
61,WFI2026-4536,167.866845,30.079459,111.735674,0.779213,0.666326,0.756942,-12.914943,0.824742,0.642650,0.549547,0.624282,0.855127
62,WFI2033-4723,134.763345,33.443944,122.518557,0.904061,0.809099,0.587003,16.479074,1.440477,1.302279,1.165489,0.845564,0.894961
63,WG0214-2105,172.587701,68.865969,145.655630,0.890399,0.788669,0.802560,1.926882,0.981781,0.874176,0.774300,0.787937,0.885748
64,WG2100-4452,124.692279,23.198296,137.406381,0.943367,0.912157,0.770706,1.836102,1.434103,1.352885,1.308127,1.105272,0.966917
